In [1]:
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
import numpy as np

# Player stats

In [2]:
player_stats = pd.read_csv('stored_datasets/All Pitchers Cleaned.csv')

In [3]:
player_stats

,Player,Debut Year,Debut Age,Retirement Year,Retirement Age,Career Length,Wins,Losses,Win Percentage,Total Decisions,...,BF,ERA+,FIP,WHIP,H9,HR9,BB9,SO9,SO/BB,Hall of Fame
0,Cy Young,1890,23,1911,44,21,511,315,0.619,826,...,29565,138,2.84,1.130,8.7,0.2,1.5,3.4,2.30,1
1,Pud Galvin,1875,18,1892,35,17,365,310,0.541,675,...,25415,107,2.96,1.191,9.6,0.2,1.1,2.7,2.43,1
2,Walter Johnson,1907,19,1927,39,20,417,279,0.599,696,...,23415,147,2.38,1.061,7.5,0.1,2.1,5.3,2.57,1
3,Phil Niekro,1964,25,1987,48,23,318,274,0.537,592,...,22677,115,3.62,1.268,8.4,0.8,3.0,5.6,1.85,1
4,Nolan Ryan,1966,19,1993,46,27,324,292,0.526,616,...,22575,112,2.97,1.247,6.6,0.5,4.7,9.5,2.04,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1210,Charlie Robertson,1919,23,1928,32,9,49,80,0.380,129,...,4453,90,3.89,1.518,10.3,0.3,3.4,2.8,0.82,0
1211,Sammy Ellis,1962,21,1969,28,7,63,58,0.521,121,...,4296,88,3.90,1.340,8.7,1.1,3.4,6.1,1.79,0
1212,Lil Stoner,1922,23,1931,32,9,50,57,0.467,107,...,4466,87,4.13,1.548,10.6,0.6,3.4,2.7,0.80,0
1213,Johnny Humphries,1938,23,1946,31,8,52,63,0.452,115,...,4342,97,3.80,1.394,9.2,0.4,3.4,2.8,0.85,0


In [41]:
player_stats.to_csv('final_tables/Player_stats.csv', index=False)

# Players and models

In [2]:
players_df = pd.read_csv('stored_datasets/All Pitchers Cleaned.csv')
results_logreg = pd.read_csv('stored_results/Results_Logreg_No_GS.csv')
results_dtree = pd.read_csv('stored_results/Results_DTree_GS_Best_F1_Stratified_KFold.csv')
results_rfc = pd.read_csv('stored_results/Results_RFC_GS_Best_Recall_Stratified_KFold.csv')
results_etc = pd.read_csv('stored_results/Results_ETC_GS_Best_Recall_Standard_KFold.csv')
results_gbc = pd.read_csv('stored_results/Results_GBC_GS_Best_Recall_Standard_KFold.csv')
results_svc = pd.read_csv('stored_results/Results_SVC_no_GS_Stratified_KFold.csv')
results_nn = pd.read_csv('stored_results/Results_Neural_Network_1.csv')
results_xgb = pd.read_csv('stored_results/Results_XGB_GS_Best_Recall_Standard_KFold.csv')

In [3]:
player_models = pd.DataFrame()
player_models['Player'] = players_df['Player']
player_models['Hall of Fame'] = players_df['Hall of Fame']

player_models['Logreg Pred'] = results_logreg['Prediction']
player_models['DTree Pred'] = results_dtree['Prediction']
player_models['RFC Pred'] = results_rfc['Prediction']
player_models['ETC Pred'] = results_etc['Prediction']
player_models['GBC Pred'] = results_gbc['Prediction']
player_models['SVC Pred'] = results_svc['Prediction']
player_models['XGB Pred'] = results_xgb['Prediction']

pred_cols = ['Logreg Pred', 'DTree Pred', 'RFC Pred', 'ETC Pred', 'GBC Pred', 'SVC Pred', 'XGB Pred']
player_models['Correct Models (out of 7)'] = player_models.apply(
    lambda row: sum(row[col] == row["Hall of Fame"] for col in pred_cols),
    axis=1
)

player_models['Mean Confidence in Prediction'] = (results_logreg['Prediction Confidence'] + results_dtree['Prediction Confidence'] + results_rfc['Prediction Confidence'] + results_etc['Prediction Confidence'] + results_gbc['Prediction Confidence'] + results_svc['Prediction Confidence']) / 6

player_models

,Player,Hall of Fame,Logreg Pred,DTree Pred,RFC Pred,ETC Pred,GBC Pred,SVC Pred,XGB Pred,Correct Models (out of 7),Mean Confidence in Prediction
0,Cy Young,1,1,1,1,1,1,1,1,7,0.966098
1,Pud Galvin,1,1,1,1,1,1,1,0,6,0.847402
2,Walter Johnson,1,1,1,1,1,1,1,1,7,0.982103
3,Phil Niekro,1,1,1,1,1,1,1,1,7,0.931801
4,Nolan Ryan,1,1,1,1,1,1,1,1,7,0.925245
...,...,...,...,...,...,...,...,...,...,...,...
1210,Charlie Robertson,0,0,0,0,0,0,0,0,7,0.992825
1211,Sammy Ellis,0,0,0,0,0,0,0,0,7,0.991213
1212,Lil Stoner,0,0,0,0,0,0,0,0,7,0.993293
1213,Johnny Humphries,0,0,0,0,0,0,0,0,7,0.992819


In [4]:
player_models['Correct Models (out of 7)'].value_counts()

Correct Models (out of 7)
7    1103
6      40
5      22
0      15
2      12
4       8
1       8
3       7
Name: count, dtype: int64

In [6]:
# Calculate Adjusted Score

dfs = [
    results_logreg,
    results_dtree,
    results_rfc,
    results_etc,
    results_gbc,
    results_svc,
    results_xgb
]

alpha = 2
beta = 2

for df in dfs:
    
    # Recreate Prediction Correct EXACTLY like before
    if {'Hall of Fame', 'Prediction'}.issubset(df.columns):
        df['Prediction Correct'] = (df['Hall of Fame'] == df['Prediction'])

    # Now calculate adjusted score (will behave correctly)
    df['Adjusted Prediction Score'] = np.where(
        df['Prediction Correct'],
        df['Prediction Confidence'] ** alpha,
        -beta * (df['Prediction Confidence'] ** alpha)
    )


In [7]:
player_models['Mean Adjusted Prediction Score'] = (results_logreg['Adjusted Prediction Score'] + results_dtree['Adjusted Prediction Score'] + results_rfc['Adjusted Prediction Score'] + results_etc['Adjusted Prediction Score'] + results_gbc['Adjusted Prediction Score'] + results_svc['Adjusted Prediction Score']) / 6
player_models

,Player,Hall of Fame,Logreg Pred,DTree Pred,RFC Pred,ETC Pred,GBC Pred,SVC Pred,XGB Pred,Correct Models (out of 7),Mean Confidence in Prediction,Mean Adjusted Prediction Score
0,Cy Young,1,1,1,1,1,1,1,1,7,0.966098,0.935399
1,Pud Galvin,1,1,1,1,1,1,1,0,6,0.847402,0.739864
2,Walter Johnson,1,1,1,1,1,1,1,1,7,0.982103,0.965164
3,Phil Niekro,1,1,1,1,1,1,1,1,7,0.931801,0.873134
4,Nolan Ryan,1,1,1,1,1,1,1,1,7,0.925245,0.862487
...,...,...,...,...,...,...,...,...,...,...,...,...
1210,Charlie Robertson,0,0,0,0,0,0,0,0,7,0.992825,0.985939
1211,Sammy Ellis,0,0,0,0,0,0,0,0,7,0.991213,0.982775
1212,Lil Stoner,0,0,0,0,0,0,0,0,7,0.993293,0.986838
1213,Johnny Humphries,0,0,0,0,0,0,0,0,7,0.992819,0.985920


In [8]:
player_models.to_csv('final_tables/Players_and_models.csv', index=False)

In [9]:
player_models_fame_only = player_models[player_models['Hall of Fame'] == 1]
player_models_fame_only.to_csv('final_tables/Players_and_models_HOF.csv', index=False)

In [10]:
player_models_no_fame_only = player_models[player_models['Hall of Fame'] == 0]
player_models_no_fame_only.to_csv('final_tables/Players_and_models_no_HOF.csv', index=False)

In [11]:
player_models_fame_only['Mean Confidence in Prediction'].mean()

np.float64(0.8349683191703432)

In [12]:
player_models_fame_only['Mean Confidence in Prediction'].median()

np.float64(0.8471287670329186)

In [13]:
player_models_no_fame_only['Mean Confidence in Prediction'].mean()

np.float64(0.9734815590839824)

In [14]:
player_models_no_fame_only['Mean Confidence in Prediction'].median()

np.float64(0.9918596504649352)

In [31]:
player_models_fame_only['Mean Adjusted Prediction Score'].mean()

np.float64(-0.13737879421921848)

In [32]:
player_models_no_fame_only['Mean Adjusted Prediction Score'].mean()

np.float64(0.9221437202287946)

# Model Comparison

In [4]:
import pandas as pd
import numpy as np

In [17]:
# Subject to change if more models are tried later. Refer to Part 4a for full list.
model_files = {
    "Logistic Regression": "stored_results/Results_Logreg_No_GS.csv",
    "Decision Tree (GS for F1, Stratified KFold)": "stored_results/Results_DTree_GS_Best_F1_Stratified_KFold.csv",
    "Random Forest (GS for Recall, Stratified KFold)": "stored_results/Results_RFC_GS_Best_Recall_Stratified_KFold.csv",
    "Extra Trees (GS for Recall, Standard KFold)": "stored_results/Results_ETC_GS_Best_Recall_Standard_KFold.csv",
    "Gradient Booster (GS for Recall, Standard KFold)": "stored_results/Results_GBC_GS_Best_Recall_Standard_KFold.csv",
    "Support Vector (no GS, Stratified KFold)": "stored_results/Results_SVC_no_GS_Stratified_KFold.csv",
    "XGBoost (GS for Recall Standard KFold)": "stored_results/Results_XGB_GS_Best_Recall_Standard_KFold.csv",
    "Neural Network": "stored_results/Results_Neural_Network_1.csv"
}

In [18]:
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
)
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score, accuracy_score, roc_auc_score
)

model_stats = []



# Define penalty weights
alpha, beta = 2, 2




for model_name, file_path in model_files.items():
    df = pd.read_csv(file_path)
    y_true = df['Hall of Fame']
    y_pred = df['Prediction']

    # Confirm binary labels
    labels = sorted(list(set(y_true) | set(y_pred)))
    print(f"{model_name} labels: {labels}")  # sanity check

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    # Optional: if you have probabilities
    y_proba = df['Prediction Confidence'] if 'y_proba' in df.columns else None


    # Use probabilities if available
    if y_proba is not None:
        roc_auc = roc_auc_score(y_true, y_proba)
    else:
        roc_auc = roc_auc_score(y_true, y_pred)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)




    # Compute adjusted prediction score if probabilities are available

    
    if 'Prediction Confidence' in df.columns:
        p = df['Prediction Confidence']
        df['Adjusted Prediction Score'] = np.where(
            df['Prediction Correct'],
            p**alpha,
            -beta * p**alpha
        )
        mean_adj_score = df['Adjusted Prediction Score'].mean()
    else:
        mean_adj_score = np.nan  # no confidence info




    

    model_stats.append({
        "Model": model_name,
        "True Negatives": tn,
        "False Positives": fp,
        "False Negatives": fn,
        "True Positives": tp,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC AUC": roc_auc,
        "Mean Adjusted Prediction Score": mean_adj_score
    })


results_df_unweighed = pd.DataFrame(model_stats)

results_df_unweighed

Logistic Regression labels: [0, 1]
Decision Tree (GS for F1, Stratified KFold) labels: [0, 1]
Random Forest (GS for Recall, Stratified KFold) labels: [0, 1]
Extra Trees (GS for Recall, Standard KFold) labels: [0, 1]
Gradient Booster (GS for Recall, Standard KFold) labels: [0, 1]
Support Vector (no GS, Stratified KFold) labels: [0, 1]
XGBoost (GS for Recall Standard KFold) labels: [0, 1]
Neural Network labels: [0, 1]


,Model,True Negatives,False Positives,False Negatives,True Positives,Accuracy,Precision,Recall,F1 Score,ROC AUC,Mean Adjusted Prediction Score
0,Logistic Regression,1117,16,34,48,0.958848,0.750000,0.585366,0.657534,0.785622,0.868560
1,"Decision Tree (GS for F1, Stratified KFold)",1101,32,32,50,0.947325,0.609756,0.609756,0.609756,0.790756,0.837889
2,"Random Forest (GS for Recall, Stratified KFold)",1123,10,31,51,0.966255,0.836066,0.621951,0.713287,0.806563,0.873379
3,"Extra Trees (GS for Recall, Standard KFold)",1110,23,22,60,0.962963,0.722892,0.731707,0.727273,0.855704,0.849992
4,"Gradient Booster (GS for Recall, Standard KFold)",1114,19,29,53,0.960494,0.736111,0.646341,0.688312,0.814786,0.884231
5,"Support Vector (no GS, Stratified KFold)",1126,7,54,28,0.949794,0.800000,0.341463,0.478632,0.667643,0.789771
6,XGBoost (GS for Recall Standard KFold),1120,13,27,55,0.967078,0.808824,0.670732,0.733333,0.829629,0.898747
7,Neural Network,1117,16,26,56,0.965432,0.777778,0.682927,0.727273,0.834403,0.018855


In [19]:
results_df_unweighed = results_df_unweighed.round(3)

In [20]:
results_df_unweighed

,Model,True Negatives,False Positives,False Negatives,True Positives,Accuracy,Precision,Recall,F1 Score,ROC AUC,Mean Adjusted Prediction Score
0,Logistic Regression,1117,16,34,48,0.959,0.750,0.585,0.658,0.786,0.869
1,"Decision Tree (GS for F1, Stratified KFold)",1101,32,32,50,0.947,0.610,0.610,0.610,0.791,0.838
2,"Random Forest (GS for Recall, Stratified KFold)",1123,10,31,51,0.966,0.836,0.622,0.713,0.807,0.873
3,"Extra Trees (GS for Recall, Standard KFold)",1110,23,22,60,0.963,0.723,0.732,0.727,0.856,0.850
4,"Gradient Booster (GS for Recall, Standard KFold)",1114,19,29,53,0.960,0.736,0.646,0.688,0.815,0.884
5,"Support Vector (no GS, Stratified KFold)",1126,7,54,28,0.950,0.800,0.341,0.479,0.668,0.790
6,XGBoost (GS for Recall Standard KFold),1120,13,27,55,0.967,0.809,0.671,0.733,0.830,0.899
7,Neural Network,1117,16,26,56,0.965,0.778,0.683,0.727,0.834,0.019


In [21]:
results_df_unweighed.to_csv('final_tables/model_comparison.csv',index=False)